In [1]:
import ast
import concurrent.futures
import glob
import itertools
import os
import pickle
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

import dask
import dask.dataframe as dd
import dask_ml.cluster as dask_cluster
from dask.distributed import Client
from dask.diagnostics import ProgressBar

from concurrent.futures import ThreadPoolExecutor
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from sklearn.linear_model import LinearRegression
from sklearn.metrics import pairwise_distances, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score
#from sklearn.cluster import KMeans

from statsmodels.regression.rolling import RollingOLS

from tqdm.notebook import tqdm
from collections import Counter
from functools import reduce
from pprint import pprint

pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### What is first `date` and `days_from_start`?

In [2]:
cutoff_to_date = pd.DataFrame(columns=["days_from_start", "date"])

mid_date = pd.to_datetime('2020-04-16')
mid_days_from_start = 86

start_date = mid_date - pd.DateOffset(days=mid_days_from_start)

new_row = {"days_from_start": 0, "date": start_date}
cutoff_to_date = cutoff_to_date.append(new_row, ignore_index=True)


new_row = {"days_from_start": mid_days_from_start, "date": mid_date}
cutoff_to_date = cutoff_to_date.append(new_row, ignore_index=True)


end_date = pd.to_datetime('2023-04-01')
num_days = (end_date - mid_date).days
end_days = num_days + mid_days_from_start

new_row = {"days_from_start": end_days, "date": end_date}
cutoff_to_date = cutoff_to_date.append(new_row, ignore_index=True)



cutoff_to_date['date'] = pd.to_datetime(cutoff_to_date['date'])

# Set 'date' column as the DataFrame index
cutoff_to_date.set_index('date', inplace=True)

df_resampled = cutoff_to_date.resample('D').ffill()
df_resampled["days_from_start"] = list(range(0, end_days+1))
df_resampled = df_resampled.reset_index()
df_resampled

,date,days_from_start
0,2020-01-21,0
1,2020-01-22,1
2,2020-01-23,2
3,2020-01-24,3
4,2020-01-25,4
...,...,...
1162,2023-03-28,1162
1163,2023-03-29,1163
1164,2023-03-30,1164
1165,2023-03-31,1165


In [3]:
df_resampled.to_csv("cutoff_to_date.csv", index=False)
